# UPENN-GBM - exploratory data analysis

[TCIA collection UPENN-GBM](https://www.cancerimagingarchive.net/collection/upenn-gbm/):
630 patients, 671 scans, advanced MRI with CaPTk radiomics, and - unlike
LUMIERE - **genuine survival censoring**.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from gbmplast.adapters import get_adapter
from gbmplast.config import load_config
from gbmplast.report import dictionary as D
from gbmplast.report.theme import apply_theme, series_color
from gbmplast.validate import validate_bundle

apply_theme()
config = load_config()
adapter = get_adapter("upenn_gbm", config)
bundle = adapter.build()
bundle.summary()

,table,n_rows,n_cols
0,patients,630,19
1,visits,671,20
2,imaging,5368,25
3,treatments,573,12
4,symptoms,0,15
5,molecular_samples,0,15
6,outcomes,1486,14


## The scan-id scheme

`UPENN-GBM-00036_11` and `UPENN-GBM-00036_21` are the **same patient**:
`_11` is the baseline pre-operative scan, `_21` a later recurrence scan.
Treating them as two patients would break grouped cross-validation.

In [2]:
visits = bundle.visits
print(visits.scan_role.value_counts().to_string())
print(f"\npatients: {visits.patient_id.nunique()}; scans: {len(visits)}")
print(f"patients with both scans: "
      f"{int((visits.groupby('patient_id').size() == 2).sum())}")

scan_role
baseline_preoperative    611
recurrence                60

patients: 630; scans: 671
patients with both scans: 41


In [3]:
visits[visits.patient_id.isin(
    visits.patient_id[visits.duplicated('patient_id', keep=False)].unique()[:4]
)][["local_patient_id", "local_visit_id", "scan_role",
    "relative_time_days", "psp_tp_score"]].sort_values("local_patient_id")

,local_patient_id,local_visit_id,scan_role,relative_time_days,psp_tp_score
35,UPENN-GBM-00036,UPENN-GBM-00036_11,baseline_preoperative,0,NaN
36,UPENN-GBM-00036,UPENN-GBM-00036_21,recurrence,521,6.0
42,UPENN-GBM-00042,UPENN-GBM-00042_11,baseline_preoperative,0,NaN
43,UPENN-GBM-00042,UPENN-GBM-00042_21,recurrence,282,6.0
46,UPENN-GBM-00045,UPENN-GBM-00045_11,baseline_preoperative,0,NaN
47,UPENN-GBM-00045,UPENN-GBM-00045_21,recurrence,113,2.0
53,UPENN-GBM-00051,UPENN-GBM-00051_11,baseline_preoperative,0,NaN
54,UPENN-GBM-00051,UPENN-GBM-00051_21,recurrence,385,6.0


## Survival has real censoring

`Survival_Status` distinguishes *Deceased* from *Alive*, *Lost to Follow-up*
and *Deceased - uncertain date of death*. Censored rows take their time from
`Survival_Censor`. This is what makes a Cox model admissible here and not on
LUMIERE.

In [4]:
outcomes = bundle.outcomes
survival = outcomes[outcomes.outcome_name == "overall_survival"]
print(survival.outcome_label.value_counts().to_string())
print(f"\nevents: {int(survival.event_indicator.sum())}, "
      f"censored: {int((survival.event_indicator == False).sum())}")

outcome_label
Deceased                              603
Alive                                  17
Lost to Follow-up                       7
Deceased - uncertain date of death      3

events: 603, censored: 27


In [5]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for i, (label, group) in enumerate(survival.groupby(survival.event_indicator)):
    ax.hist(pd.to_numeric(group.outcome_value, errors="coerce").dropna(),
            bins=40, alpha=0.7, color=series_color(i),
            label="event (deceased)" if label else "censored")
ax.set_xlabel("time from anchor scan (days)")
ax.set_ylabel("patients")
ax.set_title("Overall survival, by event status")
ax.legend()
plt.show()

## Molecular markers and their missingness

In [6]:
patients = bundle.patients
for col in ("mgmt_status", "idh_status", "sex"):
    print(f"{col}:")
    print(patients[col].value_counts(dropna=False).to_string())
    print()
print("KPS available for", int(patients.kps.notna().sum()), "patients")

mgmt_status:
mgmt_status
<NA>             327
unmethylated     160
methylated       115
indeterminate     28

idh_status:
idh_status
wildtype         514
indeterminate     97
mutant            19

sex:
sex
male      378
female    252

KPS available for 75 patients


## Imaging availability

DTI and DSC are not available for every scan; the canonical table records
that per series rather than assuming a complete protocol.

In [7]:
D.modality_completeness(bundle)

,cohort,modality,n_rows,n_available,n_unavailable,n_unknown,pct_available,n_visits_with_modality,n_patients_with_modality
0,upenn_gbm,DSC,671,534,137,0,79.58,534,497
1,upenn_gbm,DTI,671,592,79,0,88.23,592,554
2,upenn_gbm,FLAIR,671,671,0,0,100.00,671,630
3,upenn_gbm,SEG,1342,843,499,0,62.82,611,611
4,upenn_gbm,T1,671,671,0,0,100.00,671,630
5,upenn_gbm,T1CE,671,671,0,0,100.00,671,630
6,upenn_gbm,T2,671,671,0,0,100.00,671,630


## CaPTk radiomic features

11 image types / parametric maps x 3 subregions (ET, ED, NC) x two
segmentation flavours. These are **CaPTk** features, not PyRadiomics, and are
not comparable with the LUMIERE feature set.

In [8]:
features = adapter.load_captk_features(segmentation="automatic")
print("shape:", features.shape)
feature_cols = adapter.feature_columns(features)
print("feature columns:", len(feature_cols))
print("\nexamples:", feature_cols[:4], "...", feature_cols[-2:])

shape: (611, 4758)
feature columns: 4752

examples: ['T1_ET_Intensity_CoefficientOfVariation', 'T1_ET_Intensity_Energy', 'T1_ET_Intensity_InterQuartileRange', 'T1_ET_Intensity_Kurtosis'] ... ['DSC_PSR_NC_NGTDM_Strength', 'DSC_PSR_NC_LBP_Radius-1_Bins-16_LBP']


In [9]:
prefixes = pd.Series([c.split("_")[0] for c in feature_cols]).value_counts()
prefixes.head(12)

DTI      1728
DSC      1296
T1GD      432
T1        432
FLAIR     432
T2        432
Name: count, dtype: int64

## The PsP/TP score

Present for the 60 recurrence scans on a 1-6 scale. The collection page does
not publish the **direction** of the scale, so this pipeline carries it as a
covariate and never uses it as an outcome label.

In [10]:
visits[visits.psp_tp_score.notna()].psp_tp_score.value_counts().sort_index()

psp_tp_score
1.0     3
2.0     5
3.0     9
4.0     9
5.0    14
6.0    20
Name: count, dtype: int64

## Analysis set for the MGMT question

In [11]:
mgmt = outcomes[outcomes.outcome_name == "mgmt_methylation"]
baseline = visits[visits.scan_role == "baseline_preoperative"]
eligible = mgmt[mgmt.visit_id.isin(set(baseline.visit_id))]
merged = eligible.merge(features[["visit_id"]], on="visit_id", how="inner")
print(f"definite MGMT calls: {len(mgmt)}")
print(f"... on a baseline pre-operative scan: {len(eligible)}")
print(f"... with CaPTk radiomics: {len(merged)}")
eligible.outcome_label.value_counts()

definite MGMT calls: 291
... on a baseline pre-operative scan: 262
... with CaPTk radiomics: 262


outcome_label
unmethylated    151
methylated      111
Name: count, dtype: int64

In [12]:
report = validate_bundle(bundle)
report.to_frame()[["check", "table", "severity", "message"]]

,check,table,severity,message
0,primary_key_unique,patients,info,primary key ['patient_id'] unique over 630 rows
1,primary_key_unique,visits,info,primary key ['visit_id'] unique over 671 rows
2,primary_key_unique,imaging,info,primary key ['imaging_id'] unique over 5368 rows
3,primary_key_unique,treatments,info,primary key ['treatment_id'] unique over 573 rows
4,primary_key_unique,outcomes,info,primary key ['outcome_id'] unique over 1486 rows
5,visit_time_known,visits,warning,8 visits have no relative_time_days; they are ...
6,longitudinal_ordering,visits,info,visit_index consistent with time for all 630 p...
